#Задание

Используя шаблон ноутбука для распознавания видов одежды и аксессуаров из набора **fashion_mnist**, выполните следующие действия:

1. Создайте **9** моделей нейронной сети с различными архитектурами и сравните в них значения точности на проверочной выборке (на последней эпохе) и на тестовой выборке.  Используйте следующее деление: обучающая выборка - **50000** примеров, проверочная выборка - **10000** примеров, тестовая выборка - **10000** примеров.

2. Создайте сравнительную таблицу в конце ноутбука, напишите свои выводы по результатам проведенных тестов.

# Шаблон ноутбука

##Импорт библиотек

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import pandas as pd
from keras.datasets import fashion_mnist
from torch.utils.data import DataLoader, TensorDataset

##Описание базы

###База: одежда, обувь и аксессуары
- Датасет состоит из набора изображений одежды, обуви, аксессуаров и их классов.
- Изображения одного вида хранятся в numpy-массиве **(28, 28)** - **x_train, x_test**.
- База содержит **10** классов: (Футболка, Брюки, Пуловер, Платье, Пальто, Сандалии/Босоножки, Рубашка, Кроссовки, Сумочка, Ботильоны) - **y_train, y_test**.
- Примеров: train - **60000**, test - **10000**.

###Вывод примеров

In [3]:
# Загрузка датасета
(x_train_org, y_train_org), (x_test, y_test) = fashion_mnist.load_data()

# Вывод размерностей выборок
x_train, x_val = x_train_org[:50000], x_train_org[50000:]
y_train, y_val = y_train_org[:50000], y_train_org[50000:]

print(x_train.shape)
print(x_val.shape)
print(x_test.shape)

(50000, 28, 28)
(10000, 28, 28)
(10000, 28, 28)


In [4]:
x_train = x_train.astype('float32') / 255.
x_val = x_val.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

In [5]:
x_train = torch.tensor(x_train, dtype=torch.float32)
x_val   = torch.tensor(x_val, dtype=torch.float32)
x_test  = torch.tensor(x_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)


In [6]:
train_dataset = TensorDataset(x_train, y_train)
val_dataset   = TensorDataset(x_val, y_val)
test_dataset  = TensorDataset(x_test, y_test)

In [7]:
import torch.nn as nn

def create_model(units1, units2, activation1, activation2):

    activations = {
        "relu": nn.ReLU(),
        "tanh": nn.Tanh(),
        "sigmoid": nn.Sigmoid()
    }

    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, units1),
        activations[activation1],
        nn.Linear(units1, units2),
        activations[activation2],
        nn.Linear(units2, 10)
    )

    return model



In [14]:
configs = [
    (128, 64, "relu", "relu"),
    (256, 128, "relu", "relu"),
    (256, 128, "tanh", "tanh"),
    (512, 256, "relu", "relu"),
    (512, 256, "tanh", "relu"),
    (128, 64, "sigmoid", "relu"),
    (256, 64, "relu", "tanh"),
    (256, 128, "relu", "sigmoid"),
    (512, 128, "tanh", "tanh"),
]
batch_sizes = [32, 64, 128, 32, 64, 128, 32, 64, 128]

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
def train(model, loader, epochs=5):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()

    return model

In [11]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            preds = torch.argmax(out, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [15]:
results = []

for i, cfg in enumerate(configs):

    bs = batch_sizes[i]

    model = create_model(*cfg)

    train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=bs)
    test_loader  = DataLoader(test_dataset, batch_size=bs)

    model = train(model, train_loader, epochs=5)

    val_acc = evaluate(model, val_loader)
    test_acc = evaluate(model, test_loader)

    results.append([
        i+1,
        bs,
        cfg,
        val_acc,
        test_acc
    ])

In [16]:
df = pd.DataFrame(results, columns=[
    "Model",
    "Batch Size",
    "Config",
    "Val Accuracy",
    "Test Accuracy"
])
df

,Model,Batch Size,Config,Val Accuracy,Test Accuracy
0,1,32,"(128, 64, relu, relu)",0.8692,0.8654
1,2,64,"(256, 128, relu, relu)",0.8760,0.8705
2,3,128,"(256, 128, tanh, tanh)",0.8786,0.8730
3,4,32,"(512, 256, relu, relu)",0.8781,0.8730
4,5,64,"(512, 256, tanh, relu)",0.8693,0.8626
5,6,128,"(128, 64, sigmoid, relu)",0.8686,0.8591
6,7,32,"(256, 64, relu, tanh)",0.8829,0.8771
7,8,64,"(256, 128, relu, sigmoid)",0.8795,0.8730
8,9,128,"(512, 128, tanh, tanh)",0.8717,0.8651


Все 9 моделей показали достаточно высокую и стабильную точность на проверочной и тестовой выборках в диапазоне примерно от 86% до 88%. Разница между значениями Val Accuracy и Test Accuracy во всех экспериментах минимальна, что указывает на хорошую обобщающую способность моделей и отсутствие выраженного переобучения.

Наилучший результат показала модель 7 (256, 64, relu, tanh, batch_size=32) с точностью около 88.3% на валидации и 87.7% на тесте. Это свидетельствует о том, что комбинация достаточно широкой сети и небольшого batch size позволяет добиться лучшего качества.

Влияние архитектуры прослеживается: модели со средними и большими слоями (256–512 нейронов) в целом показывают более высокую точность по сравнению с более компактными конфигурациями. Однако чрезмерное увеличение размеров сети (например, модель 5) не всегда приводит к улучшению результатов, что указывает на эффект насыщения.

Функции активации также оказывают влияние. Наиболее стабильные результаты демонстрируют модели с ReLU на первом слое, тогда как использование tanh или sigmoid во всех слоях (например, модель 3 или 9) не даёт существенного выигрыша и иногда немного уступает по точности.

Влияние размера батча умеренное:
модели с batch_size = 32 (например, модель 7) показывают немного лучшее обобщение, тогда как более крупные значения (64 и 128) дают сопоставимые, но иногда чуть менее стабильные результаты.